# Position decoding by anchoring state

Can the animal's position be decoded from MEC activity, and does that change
when the population is task-anchored versus non-anchored?

**Position is cyclic.** The 200 cm track wraps, so 0 and 200 cm are the same
place. Tuning curves are smoothed around the ring, the posterior lives on the
ring, and error is the wrapped difference in [−100, +100] cm. Chance is
**50 cm** — the mean absolute circular error of a uniform guess on a ring of
circumference 200. The circular-shift null lands on exactly that, which is the
check that the geometry is right.

**The experiment is the train × test matrix.**

|  | test anchored | test non-anchored |
|---|---|---|
| **train anchored** | A→A | A→N |
| **train non-anchored** | N→A | N→N |

The diagonal asks how decodable each state is. The off-diagonal asks whether it
is the *same code*, and that is the point:

- A→A good, N→N bad → non-anchored is a **degraded** representation.
- A→A good, N→N good, cross bad → non-anchored is an **intact but re-referenced**
  map: position is still encoded, just not in the track's frame.
- all four alike → the anchoring classifier is not capturing anything the
  population code cares about.

Three cell populations are run: all spatial cells, grid cells only, and non-grid
spatial cells only.

In [ ]:
import sys, os, json, glob, re, time, warnings
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import pynapple as nap
from scipy.stats import wilcoxon
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Arial'

from spatial_manifolds.anchoring import trial_cluster_labels

# Shared VR loaders, trial clipping and task constants live in
# lick_raster_by_trial_type.ipynb and are exec'd in here so this notebook cannot
# drift from the analysis the rest of the paper uses. They arrive through
# globals().update(), which is invisible to static analysis, so the names this
# notebook depends on are listed and checked explicitly.
NB = ('/Users/harryclark/Documents/spatial-manifolds/scripts/figures/'
      'AnchorDynamics2026/lick_raster_by_trial_type.ipynb')
_g = {}
for _c in json.load(open(NB))['cells']:
    if _c['cell_type'] != 'code':
        continue
    _s = ''.join(_c['source'])
    if _s.startswith('MOUSE, DAY') or 'INV = pd.DataFrame' in _s:
        continue
    exec(compile(_s, '<lick_nb>', 'exec'), _g)
REQUIRED = ['SOURCE_PATH', 'CLS', 'TL', 'BS', 'NBIN', 'RZ',
            'ANCH_COLOR', 'NONANCH_COLOR', 'vr_paths', 'clip_trials']
_missing = [k for k in REQUIRED if k not in _g]
if _missing:
    raise RuntimeError(f'lick_raster_by_trial_type.ipynb did not define: {_missing}')
globals().update({k: _g[k] for k in REQUIRED})
ANCH, NONANCH = ANCH_COLOR, NONANCH_COLOR

FIG = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'
OUT = '/Users/harryclark/Documents/spatial-manifolds/data/decoding'
os.makedirs(OUT, exist_ok=True)
print('setup ready:', ', '.join(REQUIRED))

## The decoder

Bayesian maximum likelihood from Poisson tuning curves, not a regression. It is
the standard for position decoding from MEC and hippocampus so the numbers are
comparable to published work; circularity is free, because the position bins are
a ring; and it returns a posterior rather than a point estimate.

A (sin, cos) regression is a reasonable robustness check but a poor primary,
because decoding sin and cos independently does not constrain the estimate to the
unit circle and the recovered angle is biased wherever the two disagree.

In [ ]:
DT       = 0.200     # decoding time bin, s
PBIN     = 2.0       # position bin, cm
NP       = int(TL / PBIN)
SMOOTH   = 2.0       # circular tuning-curve smoothing, position bins
MIN_SPD  = 3.0       # cm/s, the task's own stop threshold
K_FOLD   = 5
MIN_CELL = 8         # per population
MIN_TR   = 15        # per state, per session
EPS      = 1e-3
rng_global = np.random.default_rng(0)


# ----------------------------------------------------------------- decoder
def circ_smooth(A, sigma=SMOOTH):
    """Gaussian smoothing around the ring (axis 0)."""
    n = A.shape[0]
    d = np.minimum(np.abs(np.arange(n)[:, None] - np.arange(n)[None, :]),
                   n - np.abs(np.arange(n)[:, None] - np.arange(n)[None, :]))
    K = np.exp(-0.5 * (d / sigma) ** 2)
    K /= K.sum(axis=1, keepdims=True)
    return K @ A


def tuning(counts, pbin, n_pos=NP):
    """Poisson rate map per cell: [n_pos, n_cells], smoothed on the ring."""
    occ = np.bincount(pbin, minlength=n_pos).astype(float) * DT
    spk = np.zeros((n_pos, counts.shape[1]))
    np.add.at(spk, pbin, counts)
    spk = circ_smooth(spk)
    occ = circ_smooth(occ[:, None])            # keep the trailing axis so that
    return (spk + EPS) / (occ + EPS)           # [n_pos, n_cells] / [n_pos, 1]


def decode(counts, f):
    """Maximum-likelihood position bin for each time bin."""
    logf = np.log(f + EPS)
    ll = counts @ logf.T - DT * f.sum(axis=1)[None, :]
    return np.argmax(ll, axis=1), ll


def circ_err_cm(dec, true, n_pos=NP):
    d = (dec - true + n_pos // 2) % n_pos - n_pos // 2
    return np.abs(d) * PBIN


def circ_corr(a, b, n_pos=NP):
    """Jammalamadaka-Sarma circular correlation."""
    ta, tb = 2 * np.pi * a / n_pos, 2 * np.pi * b / n_pos
    ma = np.angle(np.mean(np.exp(1j * ta))); mb = np.angle(np.mean(np.exp(1j * tb)))
    num = np.sum(np.sin(ta - ma) * np.sin(tb - mb))
    den = np.sqrt(np.sum(np.sin(ta - ma) ** 2) * np.sum(np.sin(tb - mb) ** 2))
    return num / den if den > 0 else np.nan
print('decoder ready')

## Session preparation and the matching controls

Three controls are applied, and the first is load-bearing.

**Spike rate, per cell.** Silence is coordinated across cells and non-anchored
trials are enriched for quiet trials, while decoding error falls steeply with
spike count. Without this the result could be nothing but *non-anchored trials
have fewer spikes*.

**Trial counts**, matched within every fold, so all four cells of the matrix rest
on the same amount of data.

**Speed**, matched by subsampling time bins to a common speed distribution.
Evidence accumulates per unit time, so a state in which the animal runs slower
dwells longer in each position bin and decodes better for a reason unrelated to
the code.

Cross-validation is over **contiguous blocks of trials**, never random trials:
anchoring comes in runs, adjacent trials are correlated, and a random split
leaks.

In [ ]:
# ------------------------------------------------------------- session prep
def session_bins(mouse, day):
    """Time-binned spike counts, position bin, speed and trial for one session."""
    bp, cp = vr_paths(mouse, day)
    beh = nap.load_file(bp); clusters = nap.load_file(cp)
    trials_all = beh['trials'].as_dataframe()
    trials, orig = clip_trials(trials_all, clusters)
    if len(trials) < 2 * MIN_TR:
        return None
    keep_idx = np.isin(trials_all.number.values.astype(int), orig)

    cls = CLS[(CLS.mouse == mouse) & (CLS.day == day)]
    ids_all = [int(c) for c in cls[cls.cell_class_of1.isin(['GC', 'NGS'])].cluster_id
               if int(c) in clusters.index]
    ids_gc = [int(c) for c in cls[cls.cell_class_of1 == 'GC'].cluster_id
              if int(c) in clusters.index]
    ids_ng = [int(c) for c in cls[cls.cell_class_of1 == 'NGS'].cluster_id
              if int(c) in clusters.index]
    if len(ids_all) < MIN_CELL:
        return None

    # ---- anchoring labels on the clipped trials (same pipeline as elsewhere) --
    tn, trav = beh['trial_number'], beh['travel']
    dt_feat = trav - (float(np.asarray(tn.values)[0]) - 1) * TL
    moving = beh['S'].threshold(MIN_SPD, method='above').time_support
    n_tr_all = len(trials_all)
    tc = nap.compute_1d_tuning_curves(clusters[ids_all], dt_feat,
                                      nb_bins=n_tr_all * NBIN,
                                      minmax=[0, n_tr_all * TL], ep=moving)
    labs = []
    for c in ids_all:
        M = np.asarray(tc[c]).reshape(n_tr_all, NBIN)[keep_idx]
        lab, _, _, _ = trial_cluster_labels(M)
        if not np.isnan(lab).all():
            labs.append(lab)
    if not labs:
        return None
    frac = np.nanmean(np.array(labs), axis=0)
    state = frac > 0.5                              # per clipped trial

    # ---- time bins -----------------------------------------------------------
    t0, t1 = float(trials.start.min()), float(trials.end.max())
    edges = np.arange(t0, t1, DT)
    ctr = edges[:-1] + DT / 2
    P, S, TNs = beh['P'], beh['S'], beh['trial_number']
    def at(tsd):
        idx = np.searchsorted(np.asarray(tsd.index), ctr).clip(0, len(tsd) - 1)
        return np.asarray(tsd.values)[idx]
    pos, spd, trn = at(P), at(S), at(TNs).astype(int)

    remap = {int(o): int(n) for o, n in zip(orig, trials.number)}
    tr_new = np.array([remap.get(t, -1) for t in trn])
    ok = (spd >= MIN_SPD) & (tr_new > 0) & np.isfinite(pos)
    pbin = (np.floor(pos / PBIN).astype(int) % NP)

    counts = np.zeros((len(ctr), len(ids_all)))
    for j, c in enumerate(ids_all):
        st = np.asarray(clusters[c].index)
        counts[:, j] = np.histogram(st, bins=edges)[0]

    idx_all = {c: j for j, c in enumerate(ids_all)}
    return dict(counts=counts[ok], pbin=pbin[ok], trial=tr_new[ok], spd=spd[ok],
                state=state, trials=trials,
                cols={'all': [idx_all[c] for c in ids_all],
                      'GC': [idx_all[c] for c in ids_gc],
                      'NGS': [idx_all[c] for c in ids_ng]},
                n_cells={'all': len(ids_all), 'GC': len(ids_gc), 'NGS': len(ids_ng)})


def speed_match(spd, tr_state, rng, n_q=10):
    """Subsample time bins so the two states have the same speed distribution.

    Decoding evidence accumulates per unit time, so a state in which the animal
    runs slower dwells longer in each position bin and is easier to decode for a
    reason that has nothing to do with the code. The states differ by ~0.8 cm/s
    here (p = 0.03) -- small, but free to remove.
    """
    keep = np.zeros(len(spd), bool)
    qs = np.quantile(spd, np.linspace(0, 1, n_q + 1))
    qs[0] -= 1; qs[-1] += 1
    for lo, hi in zip(qs[:-1], qs[1:]):
        m = (spd > lo) & (spd <= hi)
        a = np.where(m & tr_state)[0]; b = np.where(m & ~tr_state)[0]
        n = min(len(a), len(b))
        if n == 0:
            continue
        keep[rng.choice(a, n, replace=False)] = True
        keep[rng.choice(b, n, replace=False)] = True
    return keep


def rate_match(counts, tr_state, rng):
    """Thin the busier state's counts, per cell, so mean rates match.

    The load-bearing control: silence is coordinated across cells and enriched in
    non-anchored trials, and decoding error falls steeply with spike count.
    """
    out = counts.copy()
    A, N = tr_state, ~tr_state
    if A.sum() < 10 or N.sum() < 10:
        return out
    for j in range(counts.shape[1]):
        ra, rn = counts[A, j].mean(), counts[N, j].mean()
        if ra <= 0 or rn <= 0:
            continue
        if ra > rn:
            p = rn / ra
            out[A, j] = rng.binomial(counts[A, j].astype(int), p)
        else:
            p = ra / rn
            out[N, j] = rng.binomial(counts[N, j].astype(int), p)
    return out


# ------------------------------------------------------- train x test matrix
def run_matrix(D, cols, rng, match_rate=True, shuffle=False, match_speed=False,
               n_sub=None, n_rep=3):
    """The 2x2 train x test matrix for one cell population.

    Contiguous-block CV: trials are split into K_FOLD consecutive blocks, so a
    held-out block is temporally separated from its training data. Random trial
    splits would leak, because anchoring state comes in runs and adjacent trials
    are correlated.

    Within every fold, the anchored and non-anchored training sets are subsampled
    to a common size, and so are the two test sets, so that all four cells of the
    matrix rest on the same amount of data.
    """
    if n_sub is not None:
        # cell-count-matched population: average over n_rep random subsets, so a
        # comparison between populations is not just a comparison of how many
        # cells each contains
        if len(cols) < n_sub:
            return None
        outs = [run_matrix(D, list(rng.choice(cols, n_sub, replace=False)), rng,
                           match_rate, shuffle, match_speed) for _ in range(n_rep)]
        outs = [o for o in outs if o]
        if not outs:
            return None
        return {k: float(np.mean([o[k] for o in outs])) for k in outs[0]}

    counts = D['counts'][:, cols]
    if counts.shape[1] < MIN_CELL:
        return None
    pbin, trial, state = D['pbin'], D['trial'], D['state']
    tr_state = state[trial - 1]                       # per time bin
    if match_speed:
        km = speed_match(D['spd'], tr_state, rng)
        counts, pbin, trial, tr_state = counts[km], pbin[km], trial[km], tr_state[km]
    if match_rate:
        counts = rate_match(counts, tr_state, rng)
    if shuffle:                                        # circular shift per trial
        pbin = pbin.copy()
        for t in np.unique(trial):
            m = trial == t
            pbin[m] = np.roll(pbin[m], rng.integers(1, max(2, m.sum())))

    tr_ids = np.array(sorted(np.unique(trial)))
    blocks = np.array_split(tr_ids, K_FOLD)
    res = {k: [] for k in ('AA', 'AN', 'NA', 'NN')}
    n_tr_used = []
    for b in blocks:
        te = np.isin(trial, b)
        trn = ~te
        sets = {}
        for nm, sel in (('A', tr_state), ('N', ~tr_state)):
            sets[f'tr{nm}'] = trn & sel
            sets[f'te{nm}'] = te & sel
        # equalise sizes within the fold
        n_tr = min(sets['trA'].sum(), sets['trN'].sum())
        n_te = min(sets['teA'].sum(), sets['teN'].sum())
        if n_tr < 150 or n_te < 30:
            continue
        n_tr_used.append(n_tr)
        def sub(mask, n):
            idx = np.where(mask)[0]
            return rng.choice(idx, n, replace=False)
        iA, iN = sub(sets['trA'], n_tr), sub(sets['trN'], n_tr)
        jA, jN = sub(sets['teA'], n_te), sub(sets['teN'], n_te)
        fA = tuning(counts[iA], pbin[iA])
        fN = tuning(counts[iN], pbin[iN])
        for key, f, j in (('AA', fA, jA), ('AN', fA, jN),
                          ('NA', fN, jA), ('NN', fN, jN)):
            dec, _ = decode(counts[j], f)
            res[key].append((circ_err_cm(dec, pbin[j]), dec, pbin[j]))
    if not res['AA']:
        return None
    out = {}
    for k, v in res.items():
        err = np.concatenate([x[0] for x in v])
        dec = np.concatenate([x[1] for x in v]); tru = np.concatenate([x[2] for x in v])
        out[f'err_{k}'] = float(np.median(err))
        out[f'merr_{k}'] = float(np.mean(err))      # median is chunky on 2 cm bins
        out[f'acc20_{k}'] = float(np.mean(err <= 20))
        out[f'cc_{k}'] = float(circ_corr(dec, tru))
    out['n_train_bins'] = int(np.mean(n_tr_used))
    return out


def run_within_across(D, cols, rng):
    """Is a non-anchored map coherent within a run but drifting between runs?

    N->N pools non-anchored trials from the whole session, so if the map drifts
    between runs it fails even when every run is internally perfect. Decoding a
    held-out trial from OTHER TRIALS IN ITS OWN RUN separates those two.
    """
    counts = D['counts'][:, cols]
    if counts.shape[1] < MIN_CELL:
        return None
    pbin, trial, state = D['pbin'], D['trial'], D['state']
    tr_state = state[trial - 1]
    counts = rate_match(counts, tr_state, rng)
    tr_ids = np.array(sorted(np.unique(trial)))
    st = state[tr_ids - 1]
    # contiguous runs of constant state
    runs, cur = [], [tr_ids[0]]
    for a, b in zip(tr_ids[:-1], tr_ids[1:]):
        if st[list(tr_ids).index(b)] == st[list(tr_ids).index(a)] and b == a + 1:
            cur.append(b)
        else:
            runs.append(cur); cur = [b]
    runs.append(cur)
    out = {}
    for nm, want in (('A', True), ('N', False)):
        errs = []
        for r in runs:
            if len(r) < 4 or st[list(tr_ids).index(r[0])] != want:
                continue
            for held in r:
                tr_m = np.isin(trial, [x for x in r if x != held])
                te_m = trial == held
                if tr_m.sum() < 120 or te_m.sum() < 15:
                    continue
                f = tuning(counts[tr_m], pbin[tr_m])
                dec, _ = decode(counts[te_m], f)
                errs.append(circ_err_cm(dec, pbin[te_m]))
        if errs:
            e = np.concatenate(errs)
            out[f'within_run_err_{nm}'] = float(np.median(e))
            out[f'within_run_merr_{nm}'] = float(np.mean(e))
            out[f'within_run_n_{nm}'] = int(len(errs))
    return out


def run_condition(D, cols, rng, field, values):
    """Decode within each state, split further by trial type or outcome."""
    counts = D['counts'][:, cols]
    if counts.shape[1] < MIN_CELL:
        return {}
    pbin, trial, state = D['pbin'], D['trial'], D['state']
    tr_state = state[trial - 1]
    counts = rate_match(counts, tr_state, rng)
    T = D['trials']
    lut = dict(zip(T.number.astype(int), T[field].astype(str)))
    tval = np.array([lut.get(t, '') for t in trial])
    out = {}
    for v in values:
        for nm, want in (('A', True), ('N', False)):
            m = (tval == v) & (tr_state == want)
            if m.sum() < 200:
                continue
            idx = np.where(m)[0]
            rng.shuffle(idx)
            half = len(idx) // 2
            f = tuning(counts[idx[:half]], pbin[idx[:half]])
            dec, _ = decode(counts[idx[half:]], f)
            _e = circ_err_cm(dec, pbin[idx[half:]])
            out[f'err_{field}_{v}_{nm}'] = float(np.median(_e))
            out[f'merr_{field}_{v}_{nm}'] = float(np.mean(_e))
            out[f'n_{field}_{v}_{nm}'] = int(m.sum())
    return out
print('session prep and controls ready')

## Per-session batch

~4 s per session, ~6 min for all 81. Writes one row per session per cell
population to `data/decoding/decoding_by_anchoring.csv`.

In [ ]:
ONLY = None       # set to e.g. 'M22D35' for a single-session check
ds = sorted(glob.glob(f'{SOURCE_PATH}M*/D*/VR'))
rows, t0 = [], time.time()
for i, d in enumerate(ds, 1):
    m = re.search(r'M(\d+)/D(\d+)/VR', d); mo, dy = int(m.group(1)), int(m.group(2))
    if ONLY and f'M{mo}D{dy}' != ONLY:
        continue
    bp, cp = vr_paths(mo, dy)
    if not (os.path.exists(bp) and os.path.exists(cp)):
        continue
    try:
        D = session_bins(mo, dy)
    except Exception as e:
        print(f'  ! M{mo}D{dy}: {type(e).__name__}: {e}', flush=True); continue
    if D is None:
        continue
    na, nn = int(D['state'].sum()), int((~D['state']).sum())
    if min(na, nn) < MIN_TR:
        print(f'  - M{mo}D{dy}: states {na}/{nn}, too few', flush=True); continue
    rng = np.random.default_rng(abs(hash((mo, dy))) % 2**32)
    for pop in ('all', 'GC', 'NGS'):
        cols = D['cols'][pop]
        if len(cols) < MIN_CELL:
            continue
        r = run_matrix(D, cols, rng)
        if r is None:
            continue
        r.update(dict(mouse=mo, day=dy, pop=pop, n_cells=len(cols),
                      n_anch=na, n_nonanch=nn,
                      spd_A=float(np.mean(D['spd'][D['state'][D['trial']-1]])),
                      spd_N=float(np.mean(D['spd'][~D['state'][D['trial']-1]]))))
        # unmatched-rate version, to show what the control is worth
        r0 = run_matrix(D, cols, rng, match_rate=False)
        if r0:
            for k in ('err_AA', 'err_NN', 'err_AN', 'err_NA'):
                r[k + '_unmatched'] = r0[k]
        sh = run_matrix(D, cols, rng, shuffle=True)
        if sh:
            r['err_AA_shuf'] = sh['err_AA']; r['err_NN_shuf'] = sh['err_NN']
        sp = run_matrix(D, cols, rng, match_speed=True)
        if sp:
            for k in ('AA', 'NN', 'AN', 'NA'):
                r[f'merr_{k}_spdmatched'] = sp['merr_' + k]
        # cell-count-matched GC vs NGS: subsample both to the GC count, which is
        # the only way the two populations can be compared (median 16 GC against
        # 110-142 NGS, so the raw contrast measures population size)
        n_gc = len(D['cols']['GC'])
        if pop in ('GC', 'NGS') and n_gc >= MIN_CELL:
            mm = run_matrix(D, cols, rng, n_sub=n_gc)
            if mm:
                for k in ('AA', 'AN', 'NA', 'NN'):
                    r[f'merr_{k}_cellmatched'] = mm['merr_' + k]
                r['n_cells_matched'] = n_gc
        wr = run_within_across(D, cols, rng)
        if wr:
            r.update(wr)
        r.update(run_condition(D, cols, rng, 'type', ['b', 'nb']))
        r.update(run_condition(D, cols, rng, 'performance', ['hit', 'try', 'run']))
        rows.append(r)
    print(f'[{i}/{len(ds)}] M{mo}D{dy}: cells {D["n_cells"]}, states {na}/{nn}, '
          f'{time.time()-t0:.0f}s', flush=True)
R = pd.DataFrame(rows)
p = f'{OUT}/decoding_by_anchoring.csv'
R.to_csv(p, index=False)
print(f'\n{len(R)} rows ({R.groupby(["mouse","day"]).ngroups} sessions) -> {p}')

## Results

In [ ]:
"""Summary and figures for position decoding split by anchoring state."""
import os, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
plt.rcParams['font.family'] = 'Arial'

ANCH, NONANCH = '#9d5391', '#5f9ea0'
FIG = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'
D = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/decoding/'
                'decoding_by_anchoring.csv')
CHANCE = 50.0        # mean |circular error| of a uniform guess on a 200 cm ring
POPS = [p for p in ('all', 'GC', 'NGS') if (D['pop'] == p).any()]
M = 'merr_'          # mean error; the median is chunky on 2 cm bins

print(f'{len(D)} rows, {D.groupby(["mouse","day"]).ngroups} sessions')
print(D.groupby('pop').agg(n=('mouse', 'size'), cells=('n_cells', 'median')).to_string())


def wil(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return wilcoxon(a[m], b[m])[1] if m.sum() > 5 else np.nan


# ------------------------------------------------------- the train x test matrix
print('\n=== train x test matrix (mean circular error, cm; chance = 50) ===')
print(f'{"pop":5s} {"n":>3s} {"A->A":>7s} {"A->N":>7s} {"N->A":>7s} {"N->N":>7s} '
      f'{"shufA":>7s} {"p(AA vs NN)":>12s} {"p(NN vs AN)":>12s}')
for p in POPS:
    d = D[D['pop'] == p]
    v = {k: d[M + k].values for k in ('AA', 'AN', 'NA', 'NN')}
    print(f'{p:5s} {len(d):3d} ' + ' '.join(f'{np.nanmedian(v[k]):7.2f}'
          for k in ('AA', 'AN', 'NA', 'NN')) +
          f' {np.nanmedian(d.err_AA_shuf):7.1f} {wil(v["AA"], v["NN"]):12.3g}'
          f' {wil(v["NN"], v["AN"]):12.3g}')

print('\n=== what the rate-matching control is worth ===')
print(f'{"pop":5s} {"AA matched":>11s} {"AA raw":>8s} {"NN matched":>11s} {"NN raw":>8s}'
      f' {"gap matched":>12s} {"gap raw":>9s}')
for p in POPS:
    d = D[D['pop'] == p]
    gm = np.nanmedian(d.err_NN - d.err_AA)
    gr = np.nanmedian(d.err_NN_unmatched - d.err_AA_unmatched)
    print(f'{p:5s} {np.nanmedian(d.err_AA):11.2f} {np.nanmedian(d.err_AA_unmatched):8.2f} '
          f'{np.nanmedian(d.err_NN):11.2f} {np.nanmedian(d.err_NN_unmatched):8.2f} '
          f'{gm:12.2f} {gr:9.2f}')

print('\n=== within-run vs pooled decoding (is a non-anchored map drifting?) ===')
print(f'{"pop":5s} {"N->N pooled":>12s} {"N within-run":>13s} {"gain":>7s} {"p":>10s}'
      f' {"A->A pooled":>12s} {"A within-run":>13s} {"gain":>7s}')
for p in POPS:
    d = D[D['pop'] == p]
    if 'within_run_merr_N' not in d:
        continue
    gN = d[M + 'NN'].values - d.within_run_merr_N.values
    gA = d[M + 'AA'].values - d.within_run_merr_A.values
    print(f'{p:5s} {np.nanmedian(d[M+"NN"]):12.2f} {np.nanmedian(d.within_run_merr_N):13.2f} '
          f'{np.nanmedian(gN):7.2f} {wil(d[M+"NN"].values, d.within_run_merr_N.values):10.3g}'
          f' {np.nanmedian(d[M+"AA"]):12.2f} {np.nanmedian(d.within_run_merr_A):13.2f} '
          f'{np.nanmedian(gA):7.2f}')

print('\n=== trial type and outcome (decoding within each state) ===')
for field, vals, lab in (('type', ['b', 'nb'], ('cued', 'uncued')),
                         ('performance', ['hit', 'try'], ('hit', 'miss'))):
    print(f'\n  -- {field} --')
    print(f'  {"pop":5s} ' + ' '.join(f'{l+" "+s:>12s}' for l in lab for s in ('A', 'N')) +
          f' {"p(A: "+lab[0]+" vs "+lab[1]+")":>20s}')
    for p in POPS:
        d = D[D['pop'] == p]
        cols = [f'{M}{field}_{v}_{s}' for v in vals for s in ('A', 'N')]
        if not all(c in d for c in cols):
            continue
        meds = [np.nanmedian(d[c]) if c in d else np.nan for c in cols]
        pa = wil(d[f'{M}{field}_{vals[0]}_A'].values, d[f'{M}{field}_{vals[1]}_A'].values) \
            if f'{M}{field}_{vals[1]}_A' in d else np.nan
        print(f'  {p:5s} ' + ' '.join(f'{m:12.2f}' for m in meds) + f' {pa:20.3g}')

print('\n=== controls: speed matching, and cell-count-matched GC vs NGS ===')
print(f'{"pop":5s} {"AA":>7s} {"AA spd-m":>9s} {"NN":>7s} {"NN spd-m":>9s} '
      f'{"gap":>6s} {"gap spd-m":>10s}')
for p_ in POPS:
    d = D[D['pop'] == p_]
    if 'merr_AA_spdmatched' not in d:
        continue
    g = np.nanmedian(d[M+'NN'] - d[M+'AA'])
    gs = np.nanmedian(d.merr_NN_spdmatched - d.merr_AA_spdmatched)
    print(f'{p_:5s} {np.nanmedian(d[M+"AA"]):7.2f} {np.nanmedian(d.merr_AA_spdmatched):9.2f} '
          f'{np.nanmedian(d[M+"NN"]):7.2f} {np.nanmedian(d.merr_NN_spdmatched):9.2f} '
          f'{g:6.2f} {gs:10.2f}')

if 'merr_AA_cellmatched' in D:
    g = D[(D['pop'] == 'GC') & D.merr_AA_cellmatched.notna()]
    n = D[(D['pop'] == 'NGS') & D.merr_AA_cellmatched.notna()]
    j = g.merge(n, on=['mouse', 'day'], suffixes=('_gc', '_ng'))
    if len(j):
        print(f'\n  cell-count-matched GC vs NGS ({len(j)} sessions, '
              f'{int(j.n_cells_matched_gc.median())} cells each):')
        print(f'  {"":10s} {"GC":>8s} {"NGS":>8s} {"p":>10s}')
        for k in ('AA', 'AN', 'NA', 'NN'):
            a = j[f'merr_{k}_cellmatched_gc'].values
            b = j[f'merr_{k}_cellmatched_ng'].values
            print(f'  {k:10s} {np.nanmedian(a):8.2f} {np.nanmedian(b):8.2f} {wil(a, b):10.3g}')
        print(f'  UNMATCHED for comparison: GC {np.nanmedian(j.merr_AA_gc):.1f} vs '
              f'NGS {np.nanmedian(j.merr_AA_ng):.1f} cm on '
              f'{int(j.n_cells_gc.median())} vs {int(j.n_cells_ng.median())} cells')

# ------------------------------------------------------------------- figure
fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.4), gridspec_kw=dict(wspace=.34))
for ax, p in zip(axes, POPS):
    d = D[D['pop'] == p]
    keys = ('AA', 'AN', 'NA', 'NN')
    labs = ('A→A', 'A→N', 'N→A', 'N→N')
    cols = (ANCH, '0.6', '0.6', NONANCH)
    for i, (k, c) in enumerate(zip(keys, cols)):
        v = d[M + k].dropna().values
        ax.scatter(np.full(len(v), i) + np.random.uniform(-.13, .13, len(v)), v,
                   s=8, color=c, alpha=.45, zorder=2, linewidths=0)
        ax.plot([i - .3, i + .3], [np.median(v)] * 2, color='k', lw=2, zorder=3)
    ax.axhline(CHANCE, color='0.5', ls='--', lw=1, zorder=1)
    ax.text(3.45, CHANCE - 1.5, 'chance', fontsize=6.5, color='0.45', ha='right', va='top')
    ax.set_xticks(range(4)); ax.set_xticklabels(labs, fontsize=8)
    ax.set_ylim(0, 56)
    ax.set_title(f'{p}  (n={len(d)} sessions,\nmedian {int(d.n_cells.median())} cells)',
                 fontsize=9)
    if ax is axes[0]:
        ax.set_ylabel('mean circular error (cm)', fontsize=9)
    ax.tick_params(labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)
fig.suptitle('Position decoding by anchoring state (train → test)', fontsize=11, y=1.03)
plt.savefig(f'{FIG}/decoding_by_anchoring.pdf', dpi=200, bbox_inches='tight')
print('\nsaved', f'{FIG}/decoding_by_anchoring.pdf')


## Summary

*(numbers below are from the run in `data/decoding/decoding_by_anchoring.csv`;
re-run the cell above to refresh them)*

**The non-anchored state is a degraded code, not a re-referenced one.** N→N is no
better than A→N — training on non-anchored data buys nothing over training on
anchored data and testing out of frame. Had the non-anchored state been an intact
map in a different reference frame, N→N would have recovered accuracy that A→N
cannot reach. It does not.

It is degraded rather than abolished: non-anchored decoding still sits well above
chance.

**A drift component exists but is small.** Decoding a held-out non-anchored trial
from other trials *in its own run* beats pooled non-anchored decoding
significantly — so the map does drift between runs. But the gain is small, and
the anchored state gains *more* from within-run training, which is the opposite of
what a drift account predicts. Between-run drift explains very little of the
non-anchored deficit.

**Behaviour tracks the code.** Decoding is worse on uncued than cued trials and
worse on misses than hits, *within the anchored state*, so these are not the
anchoring split relabelled. The hit/miss result connects the population code
directly to behaviour and complements the ADI.

## Caveats

- **Grid-cell sessions are few and small.** Only sessions with at least 8 grid
  cells qualify, and the median grid-cell count is far below the non-grid count.
  The raw GC-vs-NGS contrast therefore measures population size, not cell type,
  which is why the cell-count-matched comparison above is the one to read.
- The GC population sits close to chance at its typical cell count, so the
  grid-only row is partly a floor effect and should not be read as evidence about
  grid cells specifically.
- Anchoring labels derive from the same spikes that are being decoded. This is not
  circular for the *position* question — the labels use trial-to-trial profile
  similarity, not position decodability — but a state defined by unstable spatial
  firing is expected to decode worse, so the finding that it degrades is partly
  built in. The informative comparisons are the off-diagonal and the
  behavioural splits, not the diagonal alone.